# AdShield AI — Real Public Data Profile

An honest profile of the public records the workspace actually scores. No synthetic rows are loaded. This reads the committed read-only snapshot (`data/deploy/adshield.duckdb`), so it reproduces from a clean checkout.

**Truth boundary:** these are consumer-harm *priors* (CFPB complaints, FTC aggregates), not ads and not platform enforcement labels. Nothing here estimates TikTok violation prevalence.

In [1]:
import sys, duckdb, pandas as pd
sys.path.insert(0, "..")
pd.set_option("display.width", 140)
db = duckdb.connect("../data/deploy/adshield.duckdb", read_only=True)
print(db.execute("SELECT count(*) AS scored_cases FROM ad_risk_scores").df().to_string(index=False))

 scored_cases
          956


## 1. Source mix

Every scored case traces to a declared public source.

In [2]:
print(db.execute("SELECT source, count(*) AS cases FROM ad_risk_scores GROUP BY 1 ORDER BY 2 DESC").df().to_string(index=False))

source  cases
  CFPB    956


## 2. The category degeneracy finding

The scored sample is dominated by a single risk category. This is the most important caveat in the whole project: it is a property of the ingestion filter, **not** a measurement of platform risk.

In [3]:
cat = db.execute("""
  SELECT risk_category, count(*) AS cases,
         round(100.0*count(*)/sum(count(*)) OVER (), 1) AS pct
  FROM ad_risk_scores GROUP BY 1 ORDER BY 2 DESC
""").df()
print(cat.to_string(index=False))

                                risk_category  cases  pct
Financial Scam / High-Risk Financial Services    949 99.3
                    Advertiser Integrity Risk      3  0.3
                 Uncategorized / Needs Review      2  0.2
                 Misinformation / Public Harm      1  0.1
  Health / Weight Loss / Pharmaceuticals Risk      1  0.1


## 3. Why: the CFPB product filter

Ingestion requests a fixed set of regulated financial products, so the vocabulary — and therefore the assigned category — skews almost entirely to finance. The denominator is chosen, not observed.

In [4]:
print(db.execute("SELECT product, count(*) AS cases FROM cfpb_complaints GROUP BY 1 ORDER BY 2 DESC LIMIT 10").df().to_string(index=False))

                                                                     product  cases
                         Credit reporting or other personal consumer reports    608
Credit reporting, credit repair services, or other personal consumer reports    155
                                                             Debt collection     76
                                                                    Mortgage     30
                          Money transfer, virtual currency, or money service     20
                                                 Credit card or prepaid card     19
                                                                 Credit card     17
                                                                Student loan      8
                                                            Credit reporting      7
                                                     Bank account or service      5


## 4. Score and severity distribution

Scores are review-priority values, not calibrated probabilities. Note the narrow band and the near-absence of high-severity cases — most complaint priors sit in the medium range by design.

In [5]:
print(db.execute("SELECT severity, count(*) AS cases, round(avg(risk_score),3) AS avg_score FROM ad_risk_scores GROUP BY 1 ORDER BY avg_score DESC").df().to_string(index=False))
print()
print(db.execute("SELECT round(min(risk_score),3) AS lo, round(avg(risk_score),3) AS mean, round(max(risk_score),3) AS hi FROM ad_risk_scores").df().to_string(index=False))

severity  cases  avg_score
    high      3      0.685
  medium    830      0.473
     low    123      0.340

  lo  mean    hi
0.34 0.456 0.685


## Takeaway

The current sample is a homogeneous CFPB financial-complaint backlog. Category share, feature lift, and any "trend" describe **this ingestion mix**, not TikTok prevalence. This is exactly why the dashboard labels the category chart *"not platform prevalence"* and why the Metric Diagnosis page decomposes movement into sample-mix vs within-segment rate before any threshold is touched.